# Challenge A: Resilient Release Scheduling — Falcon Reservoir
**OQI Hackathon · Junio 2026**

Este notebook implementa el benchmark oficial completo:
1. Carga y preprocesamiento de datos IBWC reales
2. Baselines clásicos (Historical Replay + Threshold Rule)
3. Formulación QUBO para optimización cuántica/híbrida
4. Solver con Simulated Annealing (proxy cuántico)
5. Comparación de SRS y análisis de escalado


In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

# ── Instalar dependencias cuánticas (descomentar según plataforma) ──
# !pip install dimod dwave-neal  # SimulatedAnnealing / D-Wave
# !pip install qiskit qiskit-optimization  # QAOA


## 1. Carga de Datos IBWC

In [ ]:
# ── Cargar datos del folder compartido del hackathon ──

def load_ibwc_csv(filepath, value_col='value'):
    df = pd.read_csv(filepath, skiprows=1,
                     parse_dates=['Timestamp (UTC-06:00)'])
    df.columns = ['timestamp'] + list(df.columns[1:])
    df = df.rename(columns={df.columns[1]: value_col})
    df = df[['timestamp', value_col]].dropna().sort_values('timestamp').reset_index(drop=True)
    return df

# Ajusta las rutas a tu entorno
storage_df  = load_ibwc_csv('DataSetExport-Total-Storage.Web-Daily-ac-ft-08461200-Instantaneous-TCM-20260622185130-7.csv', 'storage_tcm')
delta_s_df  = load_ibwc_csv('DataSetExport-Discharge-Total.Change-in-Storage-08461200-Instantaneous-TCM-20260622185956.csv', 'delta_S_tcm')
release_df  = load_ibwc_csv('DataSetExport-Discharge.Best-Available-08461300-Instantaneous-m-3-s-20260622190542-2.csv', 'discharge_m3s')

# Convertir caudal instantáneo m³/s → volumen semanal en TCM
# Resample a semanal: promedio * 7 * 86400 / 1000
release_df = release_df.set_index('timestamp')
release_weekly = release_df['discharge_m3s'].resample('W').mean() * 7 * 86400 / 1000
release_weekly = release_weekly.reset_index().rename(columns={'discharge_m3s': 'R_obs_tcm'})

# Resample storage change a semanal (suma)
delta_s_df = delta_s_df.set_index('timestamp')
delta_s_weekly = delta_s_df['delta_S_tcm'].resample('W').sum().reset_index()

# Merge por semana
weekly = pd.merge(release_weekly, delta_s_weekly, on='timestamp', how='inner')
# Storage inicial (último dato disponible)
S_init = storage_df['storage_tcm'].iloc[-1]
print(f'Semanas disponibles: {len(weekly)}')
print(f'Storage inicial: {S_init:,.0f} TCM ({S_init/storage_df["storage_tcm"].max()*100:.1f}% Smax)')
weekly.head()


## 2. Parámetros Oficiales del Benchmark

In [ ]:
# ── Seleccionar ventana temporal ──
# Usar últimas 52 semanas disponibles (o ajustar)
WINDOW = 52  # semanas para instancia Large
df = weekly.tail(WINDOW).reset_index(drop=True)
R_obs  = df['R_obs_tcm'].values
dS_obs = df['delta_S_tcm'].values

# Parámetros oficiales
S_MAX   = storage_df['storage_tcm'].max()  # capacidad total de conservación
S_MIN   = 0.25 * S_MAX                     # umbral crítico (eq. 15)
eta     = 0.10                             # restricción anti-hoarding (eq. 14)
L       = 5                                # niveles oficiales (eq. 9)

R_median_weekly = np.median(R_obs)
delta_u = 0.25 * R_median_weekly           # eq. 10
u_max   = 2 * delta_u                      # eq. 11
levels  = np.array([-2, -1, 0, 1, 2]) * delta_u  # eq. 9

# Instancias benchmark
INSTANCES = {
    'Small':  {'T': 12, 'L': 3},
    'Medium': {'T': 26, 'L': 5},  # ← OFICIAL
    'Large':  {'T': 52, 'L': 5},
}

def get_weights(T, u_max, S_min):
    w1 = 1.0 / ((T + 1) * S_min**2)
    w2 = 0.1  / (T * u_max**2)
    w3 = 0.1  / ((T - 1) * (2 * u_max)**2)
    return w1, w2, w3

T = INSTANCES['Medium']['T']  # instancia oficial
w1, w2, w3 = get_weights(T, u_max, S_MIN)
print(f'Δu = {delta_u:,.0f} TCM/semana  |  u_max = {u_max:,.0f} TCM/semana')
print(f'S_max = {S_MAX:,.0f} TCM  |  S_min = {S_MIN:,.0f} TCM')
print(f'Niveles: {levels}')


## 3. Función SRS y Baselines

In [ ]:
def compute_SRS(u_seq, S0, dS_obs, R_obs, S_min, S_max, u_max, w1, w2, w3, eta=0.10):
    """Calcula Storage Resilience Score y verifica factibilidad."""
    T = len(u_seq)
    S = S0
    C_crit, C_dev, C_smooth = 0.0, 0.0, 0.0
    feasible = True
    storages = [S0]
    for t in range(T):
        u = u_seq[t]
        R = R_obs[t] + u
        S = S + dS_obs[t] - u
        if R < 0 or abs(u) > u_max + 1e-6 or S < 0 or S > S_max:
            feasible = False
        C_crit  += max(0, S_min - S) ** 2
        C_dev   += u ** 2
        if t > 0:
            C_smooth += (u - u_seq[t-1]) ** 2
        storages.append(S)
    if abs(sum(u_seq)) > eta * sum(R_obs[:T]):
        feasible = False
    SRS = -(w1*C_crit + w2*C_dev + w3*C_smooth)
    return SRS, storages, feasible

# Datos para instancia Medium (T=26)
dS_T = dS_obs[:T]
R_T  = R_obs[:T]
args = (S_init, dS_T, R_T, S_MIN, S_MAX, u_max, w1, w2, w3)

# Baseline 1: Historical Replay (eq. 16)
u_hist = np.zeros(T)
SRS_hist, S_hist, _ = compute_SRS(u_hist, *args)

# Baseline 2: Threshold Rule (eq. 18)
def threshold_rule(S0, dS_obs, R_obs, S_min, delta_u, T):
    u, S = [], S0
    for t in range(T):
        adj = -delta_u if S < S_min else 0.0
        u.append(adj)
        S = S + dS_obs[t] - adj
    return np.array(u)

u_rule = threshold_rule(S_init, dS_T, R_T, S_MIN, delta_u, T)
SRS_rule, S_rule, feas_rule = compute_SRS(u_rule, *args)

print(f'SRS Historical Replay: {SRS_hist:.6f}')
print(f'SRS Threshold Rule:    {SRS_rule:.6f}  (ΔSRS={SRS_rule-SRS_hist:+.6f}, factible={feas_rule})')


## 4. Formulación QUBO

In [ ]:
def build_qubo(dS_obs, R_obs, S0, S_min, S_max, u_max, w1, w2, w3, levels, T, lambda_pen=1e8, eta=0.10):
    """
    One-hot encoding: x[t,k]=1 si en semana t se elige nivel levels[k]
    Variables: N = T * L
    Retorna: Q (matriz NxN), dict de índices
    """
    L = len(levels)
    N = T * L
    Q = np.zeros((N, N))
    def idx(t, k): return t * L + k

    # Restricción one-hot por paso de tiempo
    for t in range(T):
        for k in range(L):
            i = idx(t, k)
            Q[i, i] += lambda_pen * (1 - 2 * 1)  # coeff lineal del cuadrado
            for k2 in range(k+1, L):
                j = idx(t, k2)
                Q[i, j] += 2 * lambda_pen

    # Objetivo: C_dev (cuadrático, diagonal)
    for t in range(T):
        for k in range(L):
            i = idx(t, k)
            Q[i, i] += w2 * levels[k]**2

    # Objetivo: C_smooth (cross-terms entre pasos consecutivos)
    for t in range(1, T):
        for k in range(L):
            for k2 in range(L):
                i, j = idx(t, k), idx(t-1, k2)
                contrib = w3 * (levels[k] - levels[k2])**2
                Q[min(i,j), max(i,j)] += contrib

    # Nota: C_crit requiere tracking temporal del storage → se agrega como penalización
    # en el post-procesamiento o se linealiza por tramos
    return Q, N

Q_small, N_s = build_qubo(dS_obs[:12], R_obs[:12], S_init, S_MIN, S_MAX,
                           u_max, w1, w2, w3, levels[:3], 12)
Q_medium, N_m = build_qubo(dS_T, R_T, S_init, S_MIN, S_MAX,
                            u_max, w1, w2, w3, levels, T)
print(f'QUBO Small  (T=12, L=3): {N_s} variables, {N_s}x{N_s} matriz')
print(f'QUBO Medium (T=26, L=5): {N_m} variables, {N_m}x{N_m} matriz')


## 5. Solver Cuántico/Híbrido

In [ ]:
# ── Opción A: SimulatedAnnealing con dimod (proxy cuántico, gratis) ──
try:
    import dimod
    bqm = dimod.BinaryQuadraticModel.from_qubo(Q_medium)
    sampler = dimod.SimulatedAnnealingSampler()
    t0 = time.time()
    response = sampler.sample(bqm, num_reads=100, num_sweeps=1000)
    t_sa = time.time() - t0
    best = response.first.sample
    # Decodificar one-hot → secuencia u(t)
    u_qubo = []
    for t in range(T):
        chosen = [k for k in range(len(levels)) if best.get(t*len(levels)+k, 0) == 1]
        u_qubo.append(levels[chosen[0]] if chosen else 0.0)
    u_qubo = np.array(u_qubo)
    SRS_qubo, S_qubo, feas_qubo = compute_SRS(u_qubo, *args)
    print(f'[dimod SA] SRS = {SRS_qubo:.6f}  ΔSRS = {SRS_qubo-SRS_hist:+.6f}  runtime = {t_sa:.2f}s')
except ImportError:
    print('dimod no instalado. Usando Simulated Annealing manual...')

# ── Opción B: D-Wave Leap (requiere API token) ──
# from dwave.system import LeapHybridSampler
# sampler = LeapHybridSampler(token='YOUR_TOKEN')
# response = sampler.sample_qubo(Q_medium)

# ── Opción C: QAOA con Qiskit ──
# from qiskit_optimization import QuadraticProgram
# from qiskit_optimization.algorithms import MinimumEigenOptimizer
# from qiskit.algorithms import QAOA
# (ver notebook extendido QAOA)


## 6. Resultados y Reporte Final

In [ ]:
print('=' * 60)
print('REPORTE OFICIAL — Challenge A Falcon Reservoir')
print('=' * 60)
print(f'Instancia: Medium (T={T}, L={L})')
print(f'S_max = {S_MAX:,.0f} TCM | S_min = {S_MIN:,.0f} TCM')
print(f'Δu = {delta_u:,.0f} TCM/sem | u_max = {u_max:,.0f} TCM/sem')
print(f'Pesos: w1={w1:.2e} | w2={w2:.2e} | w3={w3:.2e}')
print()
print(f'{"Método":<28} {"SRS":>12} {"ΔSRS vs hist":>14} {"Factible":>10}')
print('-' * 68)
for nombre, srs in [("Hist. Replay (u=0)", SRS_hist), ("Threshold Rule", SRS_rule)]:
    print(f'{nombre:<28} {srs:>12.6f} {srs-SRS_hist:>+14.6f} {True:>10}')
print()
print('NOTA: Reemplazar con resultados del solver cuántico/híbrido')
print('Reportar adicionalmente:')
print('  - Runtime clásico vs cuántico')
print('  - Escalado: Small → Medium → Large')
print('  - Feasibility de la solución final')
